# Bronze -- `bronze_adventure_works_region`

Landing only. No deduplication, no filtering, no business logic.

**Source:** `adventure_works.region`  
**File:** `Region.csv`  
**Watermark:** `None`  
**Load pattern:** `full`

> Every upstream defect survives this layer intact. If bronze silently fixed anything, a silver bug would be indistinguishable from an upstream change and replay would not reproduce the original state.

> GENERATED FILE -- DO NOT EDIT.
Produced by framework/generators/generate_notebooks.py from the project spec set. Edit the spec and regenerate; hand edits are overwritten and will fail the notebook-lint gate.

In [ ]:
# Parameters -- overridden per environment by the deployment pipeline.
# See 05-deployment.yaml `parameterisation`.
target_item = "lh_bronze"
source_item = "lh_bronze"
environment = "dev"
dq_failure_action = "warn"

import sys
import re
from datetime import datetime

from pyspark.sql import functions as F

from ttfabric.cleansing import RuleContext, get_rule
from ttfabric.quality import DQRunLog

load_id = f"load_{datetime.utcnow():%Y%m%d_%H%M%S}"

def resolve_table(name: str):
    """Resolve a spec table reference to a DataFrame.

    Deliberately UNQUALIFIED, so the read lands in the default lakehouse.

    Rules reference tables in their OWN layer -- enforce_referential_integrity
    against dim_products, recompute_total_from_lines against fct_order_items --
    and those peers live in the item this notebook writes to, not the one it
    reads its source from. Qualifying with source_item sent them to
    lh_bronze.dim_products, which does not and should not exist.

    The single cross-item read, this table's own bronze source, is qualified
    explicitly at the call site instead.
    """
    bare = name.split(".")[-1]
    return spark.read.table(bare)

ctx = RuleContext(
    spark=spark,
    load_id=load_id,
    environment=environment,
    table="bronze_adventure_works_region",
    resolve_table=resolve_table,
    apply_masking=(environment in ("uat", "prod")),
)

dq = DQRunLog(spark, load_id=load_id, layer="bronze", table_name="bronze_adventure_works_region")
print(f"load_id={load_id}  environment={environment}  table=bronze_adventure_works_region")

In [ ]:
# ---- Declared schema ---------------------------------------------
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, LongType, BooleanType, DateType)

schema = StructType([
    StructField("sales_territory_key", IntegerType(), True),
    StructField("region", StringType(), True),
    StructField("country", StringType(), True),
    StructField("group", StringType(), True),
])

In [ ]:
# ---- Read the landed file ----------------------------------------
# Handle both tab-delimited (current format) and comma-delimited (expected format)
landing_path = "Files/bronze/adventure_works/region"

def normalize_header(name: str) -> str:
    """Convert PascalCase/spaces to snake_case."""
    name = name.replace(" ", "_").replace("-", "_")
    s1 = re.sub("(.)([A-Z][a-z]+)", r"\1_\2", name)
    result = re.sub("([a-z0-9])([A-Z])", r"\1_\2", s1).lower()
    return re.sub(r"_+", "_", result)

# Try reading with tab delimiter first (current format)
df_raw = (spark.read
    .option("header", "true")
    .option("delimiter", "\t")
    .option("encoding", "utf-8")
    .option("quote", '"')
    .csv(landing_path))

# Rename columns from PascalCase to snake_case
column_mapping = {old: normalize_header(old) for old in df_raw.columns}
df = df_raw
for old_name, new_name in column_mapping.items():
    if old_name != new_name:
        df = df.withColumnRenamed(old_name, new_name)

# Cast columns to match schema
df = df.select(
    F.col("sales_territory_key").cast(IntegerType()),
    F.col("region").cast(StringType()),
    F.col("country").cast(StringType()),
    F.col("group").cast(StringType()),
)

rows_in = df.count()
dq.record_input(rows_in)
print(f"read {rows_in:,} rows from {landing_path}")
print(f"columns: {df.columns}")

In [ ]:
# ---- Landing checks ----------------------------------------------
expected = [f.name for f in schema.fields]
actual = df.columns

if actual != expected:
    raise AssertionError(
        f"header_matches_registry failed.\n"
        f"  registry: {expected}\n"
        f"  file:     {actual}\n"
        f"Order matters -- the schema is applied positionally."
    )
if rows_in == 0:
    raise AssertionError("row_count_not_zero failed: a zero-row file "
                         "almost always means a broken export")
print("landing checks passed")

In [ ]:
# ---- Audit columns and write -------------------------------------
out = (df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source", F.lit("adventure_works"))
    .withColumn("_load_id", F.lit(load_id))
    .withColumn("_source_file", F.input_file_name())
)

out = out.withColumn("ingest_date", F.current_date())

(out.write.mode("overwrite")
    .option("partitionOverwriteMode", "dynamic")
    .partitionBy("ingest_date")
    .format("delta").saveAsTable("bronze_adventure_works_region"))

dq.record_output(rows_in)
print(f"landed {rows_in:,} rows into bronze_adventure_works_region")
dq.flush()